# 06 · Dataset & Transforms

**Purpose.** Prepare fast image loading (resized cache) and check, by eye, what the model will actually see.

| | |
|---|---|
| **Reads** | `data/splits/manifest_split.csv` |
| **Writes** | `data/cache/*.jpg`, `cache_path` column in the split CSVs, figures |
| **Run time (Colab T4)** | 15-30 min the first time (resizes every image once) |
| **Next notebook** | 07 · Baseline Training |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

## 1. Build the resized-image cache

Some originals are 4,608 px wide. Decoding them every epoch from Drive would make each epoch take
many minutes. We shrink every image **once** so its shorter side is 256 px and save it to
`data/cache/`. Training reads these small files, then crops 224 px.

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
from dermalens.data import build_transforms, cache_resized_images, denormalize, make_loader, SkinDataset, load_rgb
from dermalens.utils import load_json

TARGET_CLASSES = load_json(paths.splits / "class_info.json")["classes"]
split = pd.read_csv(paths.splits / "manifest_split.csv", dtype={"phash": str})
split = cache_resized_images(split, paths.raw, paths.cache, short_side=CONFIG["data"]["cache_short_side"])

for name in ("train", "val", "test"):                  # save cache_path into the frozen splits
    split[split["split"] == name].to_csv(paths.splits / f"{name}.csv", index=False)
split.to_csv(paths.splits / "manifest_split.csv", index=False)
print("Cached images:", len(list((paths.cache / f"s{CONFIG['data']['cache_short_side']}").glob("*.jpg"))))

### Optional on Colab: copy the cache to the local disk

Reading from the Colab machine's own disk is much faster than reading from Drive. The copy lasts
until the session ends; notebooks 07-11 use it automatically when it exists.

In [ ]:
import shutil
from pathlib import Path

LOCAL_CACHE = Path("/content/cache")
if IN_COLAB and not LOCAL_CACHE.exists():
    shutil.copytree(paths.cache, LOCAL_CACHE)
IMAGE_ROOT = LOCAL_CACHE if LOCAL_CACHE.exists() else paths.cache
print("Training will read images from:", IMAGE_ROOT)

## 2. Dataset sanity checks

In [ ]:
image_size = CONFIG["data"]["image_size"]
train = split[split["split"] == "train"]
dataset = SkinDataset(train, IMAGE_ROOT, TARGET_CLASSES, build_transforms(image_size, train=True))

tensor, meta_vector, class_index = dataset[0]
print("Metadata vector (21 numbers: age one-hot+missing, sex+missing, 11 regions+missing):", meta_vector.tolist())
print("Images in train:", len(dataset))
print("One sample:", tuple(tensor.shape), "| values from", round(float(tensor.min()), 2), "to", round(float(tensor.max()), 2))
print("Its class:", class_index, "=", TARGET_CLASSES[class_index])
print("Class index map:", {name: i for i, name in enumerate(TARGET_CLASSES)})

## 3. What augmentation does

Each row: the cached photo, then four random training versions of it. They should still clearly
show the lesion, with natural colours. If an augmentation hides the lesion or changes skin colour,
it is too strong.

In [ ]:
augment = build_transforms(image_size, train=True)
fig, axes = plt.subplots(len(TARGET_CLASSES), 5, figsize=(12, 2.6 * len(TARGET_CLASSES)))
for row, label in enumerate(TARGET_CLASSES):
    item = train[train["label"] == label].iloc[0]
    original = load_rgb(IMAGE_ROOT / item["cache_path"])
    axes[row, 0].imshow(original)
    axes[row, 0].set_title(label, fontsize=9)
    for col in range(1, 5):
        axes[row, col].imshow(denormalize(augment(original)).permute(1, 2, 0))
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(paths.figures / "augmentation_preview.png", dpi=120)
plt.show()

## 4. How fast can we feed the GPU?

In [ ]:
loader = make_loader(train, IMAGE_ROOT, TARGET_CLASSES, image_size, train=True,
                     batch_size=CONFIG["train"]["batch_size"], num_workers=CONFIG["train"]["num_workers"])
started, images_seen = time.time(), 0
for batch_number, (images, metas, targets) in enumerate(loader):
    images_seen += len(targets)
    if batch_number == 4:
        break
rate = images_seen / (time.time() - started)
print(f"{rate:.0f} images/second -> one epoch of {len(train)} images ≈ {len(train) / rate:.0f} s of loading")